This notebook is used to get the datasets that need to be updated after the HxGN offset of the data is done.

Steps include:
1. For each MC Project in B1, fetch the datasets connected to it.
2. Find the latest version of the datasets in each project
3. Fetch the dataset details from TDEI system
4. Ensure that the latest dataset is linked 
5. Churn out the following information:
    1. Name
    2. Area
    3. Dataset ID
    4. Dataset Name
    5. Dataset Version
    6. Service Name where the Dataset is located


In [2]:
# import pymongo
from pymongo import MongoClient
import os
from dotenv import load_dotenv
load_dotenv()
client = MongoClient(os.environ.get('MONGO_CONNECTION'))
db = client['wa-proviso-2']
projects_collection = db['projects']

pipeline = [
    {
        '$match': {
            'biennium': {
                '$exists': False
            }
        }
    }, {
        '$project': {
            'name': 1, 
            'area': 1, 
            '_id': 1
        }
    }, {
        '$addFields': {
            'project_id_str': {
                '$toString': '$_id'
            }
        }
    }, {
        '$lookup': {
            'from': 'tdei_datasets', 
            'localField': 'project_id_str', 
            'foreignField': 'project_id', 
            'as': 'datasets'
        }
    }
]


/var/folders/2s/p3bg9q7n54d7kq9h25t04jph0000gp/T/ipykernel_73416/3288824598.py:6: UserWarning: You appear to be connected to a CosmosDB cluster. For more information regarding feature compatibility and support please visit https://www.mongodb.com/supportability/cosmosdb
  client = MongoClient(os.environ.get('MONGO_CONNECTION'))


In [70]:
result = projects_collection.aggregate(pipeline,batchSize=10)
projects_data_full = []
for single_result in result:
    name = single_result['name']
    area = single_result['area']
    project_id_str = single_result['project_id_str']
    datasets_info = []
    for dataset in single_result['datasets']:
        tdei_id = dataset['tdei_dataset_id']
        project_group = dataset['project_group']['name']
        service = dataset['service']['name']
        dataset_name = dataset['metadata']['dataset_detail']['name']
        version = dataset['metadata']['dataset_detail']['version']
        datasets_info.append({
            'tdei_dataset_id':tdei_id,
            'project_group':project_group,
            'service':service,
            'dataset_name':dataset_name,
            'version':version
        })
    project_data = {
        'name':name,
        'area':area,
        'datasets':datasets_info,
        'project_id_str':project_id_str
    }
    projects_data_full.append(project_data)
        
    
    

In [71]:
print(len(projects_data_full))

372


In [72]:
import pandas as pd

projects_df = pd.DataFrame(projects_data_full)

In [73]:
projects_df.columns

Index(['name', 'area', 'datasets', 'project_id_str'], dtype='object')

In [74]:

def get_latest_version(datasets):
    latest_version = None
    for dataset in datasets:
        version = dataset['version']
        if version is None:
            continue
        version_float = float(version)
        if latest_version is None:
            latest_version = version_float
            continue
        if latest_version < version_float:
            latest_version = version_float
    return latest_version

projects_df['latest_version'] = projects_df['datasets'].apply(get_latest_version)


In [75]:
projects_df['latest_version'].value_counts()

latest_version
1.5    129
1.4     68
1.7     32
1.2     31
1.6     28
1.8     19
1.9     12
1.3     11
2.0      7
2.1      4
2.2      1
Name: count, dtype: int64

In [76]:
def get_latest_tdei_dataset_id(row):
    latest_version = row.get('latest_version')
    target_id = None
    if latest_version is not None :
        datasets = row.get('datasets')
        match = next((item for item in datasets if float(item["version"]) == latest_version), None)
        if match is not None:
            target_id = match.get('tdei_dataset_id')
    return target_id

def get_service_name(datasets):
    if datasets is None:
        return None
    if len(datasets) == 0:
        return ''
    first_element = datasets[0]
    return first_element.get('service','')


projects_df['latest_dataset_id'] = projects_df.apply(get_latest_tdei_dataset_id,axis=1)
projects_df['service_name'] = projects_df['datasets'].apply(get_service_name)

In [77]:
len(projects_df)

372

In [78]:
projects_df['service_name'].value_counts()

service_name
WA Proviso              186
GS_WA_Proviso           117
Proviso_Unions           36
                         30
AU Test                   2
TDEI Columbia County      1
Name: count, dtype: int64

In [79]:
projects_df.columns

Index(['name', 'area', 'datasets', 'project_id_str', 'latest_version',
       'latest_dataset_id', 'service_name'],
      dtype='object')

In [83]:
proviso_datasets = projects_df[projects_df['service_name']=='Proviso_Unions']

In [86]:
len(proviso_datasets)
# reset the index
proviso_datasets.reset_index()

,index,name,area,datasets,project_id_str,latest_version,latest_dataset_id,service_name
0,329,GS Asotin County,17.854347,[{'tdei_dataset_id': '88a8e7ed-5534-4c8e-b727-...,6848071f7fe063e3230cab6b,1.3,d463550b-6e4d-4a06-8f3b-2e3feffd586c,Proviso_Unions
1,330,GS Adams County,24.404723,[{'tdei_dataset_id': '76d894dd-000e-4e36-b0b0-...,684862347fe063e3230cab9a,1.4,40d956fb-6533-48c1-8a4c-26d6506eb4d1,Proviso_Unions
2,333,GS Clallam County,132.867548,[{'tdei_dataset_id': '2c8946f0-dea8-48a8-badd-...,684909387fe063e3230cabbb,1.2,7b51835e-9c44-4ec6-b4d2-241a922d1467,Proviso_Unions
3,334,GS Clark County,483.626626,[{'tdei_dataset_id': '71467d7c-fba7-4737-9fda-...,684909617fe063e3230cabbd,1.2,8084d987-9c11-496e-8b77-d4effc9d3f52,Proviso_Unions
4,335,GS Columbia County,3.291232,[{'tdei_dataset_id': '076f595d-d9ee-4f1f-b297-...,6849098b7fe063e3230cabbf,1.2,d88207ab-2b6b-4e07-ad07-d2f622657ba4,Proviso_Unions
5,336,GS Cowlitz County,131.568540,[{'tdei_dataset_id': '10facfd9-8fb4-43eb-8f1b-...,684909ac7fe063e3230cabc1,1.2,1955dfe6-7334-46c3-ab4e-10ef139419b0,Proviso_Unions
6,337,GS Douglas County,42.886803,[{'tdei_dataset_id': '5755b818-f4a5-42d6-9a77-...,68490b457fe063e3230cabc3,1.3,45d8b021-3ec8-4086-9767-a392bb275e15,Proviso_Unions
7,338,GS Ferry County,9.763011,[{'tdei_dataset_id': '29210983-776a-4113-91c3-...,68490bc87fe063e3230cabc5,1.2,4fb55700-07c3-4987-bd41-d2af827cfe87,Proviso_Unions
8,339,GS Franklin County,138.180325,[{'tdei_dataset_id': '2b04b000-4cbf-4bf8-8ed9-...,68490c0c7fe063e3230cabc7,1.2,02f87590-0943-4b48-b668-0ae205d9099d,Proviso_Unions
9,340,GS Garfield County,3.650289,[{'tdei_dataset_id': '9822620c-079c-4f33-9069-...,68490c307fe063e3230cabc9,1.2,aff6af6e-7b81-44b4-bb7e-015fb32179c5,Proviso_Unions


In [87]:
proviso_datasets[['name','area','latest_version','latest_dataset_id','project_id_str']].to_csv('Proviso_Unions.csv')

In [ ]:
result = projects_collection.find({'biennium':{'$exists':False}},{'_id':1,'name':1,'area':1})
datasets_collection = db['tdei_datasets']
projects_info = []
for single in result:
    projects_info.append(single)
    print(str(single['_id']))
    project_id = str(single['_id'])
    datasets_query = datasets_collection.find({'project_id':project_id})
    print(datasets_query)
print(len(projects_info))
    

SyntaxError: invalid syntax (318532354.py, line 8)